In [ ]:
from pathlib import Path
import os
import glob
from importlib import reload

import jax, jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "core").exists() and (REPO_ROOT.parent / "core").exists():
    os.chdir(REPO_ROOT.parent)
    REPO_ROOT = Path.cwd()

from core import config, data, infer, net, pipelines, train

reload(pipelines); reload(infer); reload(train);

In [ ]:
mt_sim_mode = 'isar2_c'
# options: 'isar2_c', 'expm_bmmat'
cest_stdup_red = 1

do_train = True
do_amide = False

saturating_readout = False
drop_first = True

patient_name = 'ichilov_alz' # 'erlangen'  ichilov_alz
sli = 27
do_all_points = True   # set True to compare posteriors across a whole slide.

In [ ]:
import datetime
timestamp = datetime.datetime.now().strftime("_%Y-%m-%d_%Hh") #%M%S")
results_folder = f'{REPO_ROOT}/figs/patients/{patient_name}_{timestamp}'
train_ckpt_folder = f'{REPO_ROOT}/ckpts/patients/{patient_name}_{timestamp}' 
test_ckpt_folder = train_ckpt_folder 

In [ ]:
config.Config.configure_AMIDE_clinical_whole_brain()    

pipelines.pipeline_config.mt_sim_mode = mt_sim_mode
pipelines.pipeline_config.cest_stdup_red = cest_stdup_red
pipelines.pipeline_config.mt_patience = 5000
pipelines.pipeline_config.mt_steps = 5000
pipelines.pipeline_config.cest_patience = 5000
pipelines.pipeline_config.cest_steps = 5000

seq_df_mt = data.read_sequence('mt', drop_first=drop_first)
seq_df_amide = data.read_sequence('amide', drop_first=drop_first)

In [ ]:
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")

In [ ]:

from core import simulation
if saturating_readout:
    simulation.sim_config.t_flip_period = 1e-3
    simulation.sim_config.flip_angle = flip_angle = 90 * jnp.pi / 180
    simulation.sim_config.flip_cos = jnp.cos(flip_angle)
    simulation.num_flip_pulses = 1

### Creating setup & inspecting the data

In [ ]:
from helpers import loaders

mt_signal, amide_signal, T1ms, T2ms, B0ppm, B1fac, roi_mask_nans = loaders.get_by_name(patient_name)()
loaders.viz(mt_signal, amide_signal, T1ms, T2ms, B0ppm, B1fac, roi_mask_nans, sli)
if drop_first and mt_signal.shape[0]>30:
    mt_signal = mt_signal[1:]
    amide_signal = amide_signal[1:]
    
data_feed_mt, data_feed_amide = [data.SlicesFeed.from_args(
    seq_name=seq_name, drop_first=drop_first,
    shape=T1ms.shape,       
    signal=signal,        
    roi_mask_nans=roi_mask_nans,
    T1a_ms=T1ms, T2a_ms=T2ms,
    B0_shift_ppm_map=B0ppm, B1_fix_factor_map=B1fac
) for seq_name, signal in [('mt', mt_signal), ('amide', amide_signal)]]    

### Training / fitting with the PdVAE


In [ ]:
import shutil 

if do_train:
    shutil.rmtree(f'{train_ckpt_folder}/mt', ignore_errors=True)
    if do_amide:
        shutil.rmtree(f'{train_ckpt_folder}/cest', ignore_errors=True)
    predictor_mt, predictor_amide = pipelines.run_train(
        datafeed_mt=data_feed_mt, 
        datafeed_cest=data_feed_amide, 
        ckpt_folder=train_ckpt_folder,
        do_cest=do_amide,
    )
    
else:
    predictor_mt = net.load_ckpt(folder=f'{test_ckpt_folder}/mt')[0]    
    predictor_amide = net.load_ckpt(folder=f'{test_ckpt_folder}/cest')[0] if do_amide else None

### Analyzing the resultant maps of tissue parameter estimates

In [ ]:
transfer_res, err_mt, err_amide = pipelines.transfer_and_plot(
    brain2test_mt=data_feed_mt, brain2test_cest=data_feed_amide,
    predictor_mt=predictor_mt, predictor_cest=predictor_amide,
    figsfolder=results_folder, figsfx='mt', do_boxplots=False, slices=range(1, 12, 2) 
)


In [ ]:
import os
os.makedirs(f'{results_folder}/intermediates', exist_ok=True)
mt_tissue_params_est = transfer_res[1] 
nn_pred_sim_signal_mt = transfer_res[2]
np.savez(
    f'{results_folder}/intermediates/mt_tissue_params_est.npz',
    mt_nn_pred_signal_normed_np=nn_pred_sim_signal_mt,    
    **mt_tissue_params_est
    )    

amide_tissue_params_est = transfer_res[4] 
nn_pred_sim_signal_amide = transfer_res[5]
if do_amide:
    np.savez(
        f'{results_folder}/intermediates/amide_tissue_params_est.npz',
        amide_nn_pred_signal_normed_np=nn_pred_sim_signal_amide,    
        **amide_tissue_params_est
        )

In [ ]:
from helpers import utils
import matplotlib.colors as mcolors

# Visualize the error maps in a slightly more informative way
plt.figure(figsize=(10, 4))

log_bins = np.logspace(np.log10(1.), np.log10(14), num=12)/100  # logarithmic
# log_bins = np.arange(0., 0.14, 0.01)  # Similar to R^2 coloring in paper
norm = mcolors.BoundaryNorm(log_bins, ncolors=plt.get_cmap('hot_r').N, clip=True)
plt.subplot(1,2,1)
cmap_nrmse = plt.cm.get_cmap("YlOrRd").copy();  cmap_nrmse.set_bad('1.0')
img = plt.imshow(err_mt[:,sli,:], norm=norm, cmap=cmap_nrmse) #'hot_r'); # plt.colorbar(img0)  # vmin=0, vmax=0.1,
utils.remove_spines(plt.gca())
cbar = plt.colorbar(img)
cbar.set_ticks(log_bins)
cbar.set_ticklabels([f"{v*100:.1f}" for v in log_bins], fontsize=12)
cbar.set_label('NRMSE (%)')

if do_amide:
    plt.subplot(1,2,2)
    utils.remove_spines(plt.gca())
    img = plt.imshow(err_amide[:,sli,:], norm=norm, cmap=cmap_nrmse)
    cbar = plt.colorbar(img)
    cbar.set_ticks(log_bins)
    cbar.set_ticklabels([f"{v*100:.1f}" for v in log_bins], fontsize=12)
    cbar.set_label('NRMSE (%)')

In [ ]:
import analyze_uncertainty as au; reload(au); reload(au.viz)
import matplotlib.patches as patches

mt_tissue_params_est = transfer_res[1] 
f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(mt_tissue_params_est, data_feed_mt.shape, is_cest=False)

crop=5    
axes = au.plot_CI_maps(
    f_est[crop:,sli,:-crop], f_sigma[crop:,sli,:-crop], 
    k_est[crop:,sli,:-crop], k_sigma[crop:,sli,:-crop], 
    is_cest=False, vmax_k=70, vmax_f=30, figsize=(10, 7), ktop=True, 
    ticks_for_all_cbars=True, kwargs_plot_map={'old_style': True, 'pad': 0.15}
)


...old cute cells been here at Apr15

perhaps doing [[20,20], [30,30], [40,40]]

In [ ]:
few_points = [[25, 25],[30, 30],[35, 35],[40, 40],[45,45],[50, 50], [55, 55], [60, 60], [65, 65], [70, 70]]
mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings,\
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics = \
        au.compare_posteriors(
            few_points, data_feed_mt, mt_tissue_params_est, seq_df=seq_df_mt,
            mt_sim_mode=mt_sim_mode, sli=sli, voxels2sample=500, visualize=True
        )

In [ ]:
import os
reload(au)
reload(au.metrics)

def meshgrid_xy_pairs(shape):
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    xy_pairs = np.stack([x, y], axis=-1).reshape(-1, 2)
    return xy_pairs.tolist()

# ALL!
ALL_xy_points = meshgrid_xy_pairs(data_feed_mt.shape[0::2])

xy_points = ALL_xy_points[::10] if not do_all_points else ALL_xy_points
print(len(xy_points))    

mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings,\
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics = \
        au.compare_posteriors(
            xy_points, data_feed_mt, mt_tissue_params_est, seq_df=seq_df_mt,
            mt_sim_mode=mt_sim_mode, sli=sli, voxels2sample=10000, visualize=False
        )
        
os.makedirs(results_folder+'/intermediates', exist_ok=True)

np.savez_compressed(
    results_folder+'/intermediates/MT_UQ_stats_allpoints.npz',
    mahas1=mahas1, mahas2=mahas2,
    CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
    good_xy_points=good_xy_points, timings=timings, train_time=0,
    f_est=f_est, k_est=k_est, cov_nnpred_scaled=cov_nnpred_scaled,
    covs_of_exact_posterior=covs_of_exact_posterior, 
    modes_of_exact_posterior=modes_of_exact_posterior, means_of_exact_posterior=means_of_exact_posterior,    
    exact_posterior_min_nrmse_percs=exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs=nn_posterior_min_nrmse_percs,
    **extra_metrics
)

In [ ]:
au.summarize_mahas(mahas1, mahas2, good_xy_points) 
plt.xlabel('Mahalanobis Distance')
for format in ['svg', 'png']:       
    plt.savefig(results_folder+f'/MT_mahas_hist.{format}', bbox_inches='tight', dpi=300, format=format)

In [ ]:
# Multiple ellipses blabla been here pre 17Dec2025

### Amide

In [ ]:
if not do_amide:
    import sys
    sys.exit(0)

In [ ]:
amide_tissue_params_est = transfer_res[4] 
f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(amide_tissue_params_est, data_feed_mt.shape, is_cest=True)
    

In [ ]:
axes = au.plot_CI_maps(
    f_est[crop:-crop,sli,crop:-crop], f_sigma[crop:-crop,sli,crop:-crop], 
    k_est[crop:-crop,sli,crop:-crop], k_sigma[crop:-crop,sli,crop:-crop], 
    is_cest=True, vmax_k=400, vmax_f=0.7, figsize=(10, 7), ktop=True, ticks_for_all_cbars=True
)


In [ ]:
few_points = [[25, 25],[30, 30],[35, 35],[40, 40],[45,45],[50, 50], [55, 55], [60, 60], [65, 65], [70, 70]]
mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings,\
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics = \
        au.compare_posteriors(
            few_points, data_feed_mt, mt_tissue_params_est, seq_df=seq_df_amide,
            is_cest=True, seq_name="amide", data_feed_cest=data_feed_amide, 
            cest_tissue_params_est=amide_tissue_params_est,
            mt_sim_mode=mt_sim_mode, sli=sli, voxels2sample=500, visualize=True
        )

In [ ]:
do_all_points = False  # set false to reduce to few min run
xy_points = ALL_xy_points[::10] if not do_all_points else ALL_xy_points

mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings,\
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics = \
        au.compare_posteriors(
            xy_points, data_feed_mt, mt_tissue_params_est, seq_df=seq_df_amide,
            is_cest=True, seq_name="amide", data_feed_cest=data_feed_amide, 
            cest_tissue_params_est=amide_tissue_params_est,
            mt_sim_mode=mt_sim_mode, sli=sli, voxels2sample=500, visualize=True
        )
        
np.savez_compressed(
    results_folder+'/intermediates/AMIDE_UQ_stats_allpoints.npz',
    mahas1=mahas1, mahas2=mahas2,
    CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
    good_xy_points=good_xy_points, timings=timings, train_time=0,
    f_est=f_est, k_est=k_est, cov_nnpred_scaled=cov_nnpred_scaled,
    covs_of_exact_posterior=covs_of_exact_posterior, 
    modes_of_exact_posterior=modes_of_exact_posterior, means_of_exact_posterior=means_of_exact_posterior,    
    exact_posterior_min_nrmse_percs=exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs=nn_posterior_min_nrmse_percs
)

In [ ]:
au.summarize_mahas(mahas1, mahas2, good_xy_points)

local redefinition of compare_posteriors and various stuff been here at 2026-Apr-15